# 08 — Experiment 4c

**Variable under test:** DPO training

**Config:** DPO pass — DPOTrainer, r=16, alpha=32, beta=0.1, 3 epochs, LR 5e-6, batch 4 (1×grad_accum 4), max_seq_length=2048, 4-bit, vision layers unfrozen.

**Dataset/Model** 04b checkpoint model + 50 training golds from dataset.marked.json + failing held-out outputs from Exp 4/4b runs

**Inputs** ~75-100 (gold, rejected) preference pairs from 04b

**What to watch for:** model binds spatial segments in stacked bar charts correctly. No regression on exp4b's 5/5/5. internal consistency (no 'tooltip visible' / 'no tooltip' contradictions in same output)

**If/Then** → If DPO loss stays near ln(2), preference signal isn't strong enough. Will need to investigate beta.

In [1]:
!mkdir -p /mnt/civicinsight/standardized /mnt/civicinsight/training /mnt/civicinsight/test /mnt/civicinsight/checkpoints/exp4c-dpo /mnt/civicinsight/results
!ls -al /mnt/civicinsight/

total 12
drwxr-xr-x 1 root root   85 Apr 26 06:33 .
drwxr-xr-x 3 root root   47 Apr 26 04:24 ..
drwxr-xr-x 1 root root   28 Apr 26 02:22 checkpoints
drwxr-xr-x 1 root root    0 Apr 26 06:33 hf_cache
drwxr-xr-x 1 root root  145 Apr 23 13:23 model
-rw-r--r-- 1 root root 6257 Apr 26 05:31 requirements-exp4c-WORKING.txt
drwxr-xr-x 1 root root   18 Apr 23 14:29 results
drwxr-xr-x 1 root root 1548 Apr 23 13:11 standardized
drwxr-xr-x 1 root root  103 Apr 23 13:31 test
drwxr-xr-x 1 root root   19 Apr 23 13:11 training


In [2]:
%%capture
%uv pip install unsloth
%uv pip install pillow==11.3.0
%uv pip install --upgrade transformers
%uv pip install trl
# Pin PEFT to match the version that wrote the SFT adapter (exp4 adapter_config.json: 0.19.1)
%uv pip install "peft==0.19.1"


In [3]:
# imports — mirrors exp4 pattern, plus trl DPO bits and peft helpers for manual SFT weight loading

from unsloth import FastVisionModel          # MUST be first
from trl import DPOTrainer, DPOConfig
from peft import set_peft_model_state_dict
from safetensors.torch import load_file as safe_load
import torch
import os
import json
import re
import random
from PIL import Image
from huggingface_hub import login, HfApi, create_repo
import time

# from 01-zeroshot tests
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

random.seed(42)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in via Modal Secret")
else:
    print("HF_TOKEN not found in environment. Add it as a Modal Secret.")


Logged in via Modal Secret


In [5]:
SFT_CHECKPOINT = "/mnt/civicinsight/checkpoints/exp4-visionunfrozen/checkpoint-65"

# Load 4-bit base — exact same call as exp4 cell 6
model, tokenizer = FastVisionModel.from_pretrained(
    "/mnt/civicinsight/model",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Attach a FRESH LoRA adapter using exp4's exact config.
# We do this instead of PeftModel.from_pretrained because the latter rejects
# Gemma4ClippableLinear at module-resolution time (it lands on the wrapper
# instead of descending to the inner .linear). FastVisionModel.get_peft_model
# goes through Unsloth's wrapper which handles the descent correctly. Then we
# copy the SFT-trained weights into the freshly-attached adapter.
model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print(f"Fresh adapter attached. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Trainable params (fresh, before SFT load):")
model.print_trainable_parameters()

# Now copy SFT-trained LoRA weights into the freshly-attached adapter.
# Adapter shapes match because we used identical r/alpha/target_modules to exp4.
sft_weights_path = os.path.join(SFT_CHECKPOINT, "adapter_model.safetensors")
sft_state = safe_load(sft_weights_path)
print(f"\nLoaded {len(sft_state)} tensors from {sft_weights_path}")

load_result = set_peft_model_state_dict(model, sft_state)

print(f"\nGPU after SFT weight load: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Trainable params (after SFT load — should be unchanged from above):")
model.print_trainable_parameters()


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.6.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Base loaded. GPU: 10.0 GB
Fresh adapter attached. GPU: 10.1 GB
Trainable params (fresh, before SFT load):
trainable params: 39,403,520 || all params: 7,980,504,352 || trainable%: 0.4937

Loaded 740 tensors from /mnt/civicinsight/checkpoints/exp4-visionunfrozen/checkpoint-65/adapter_model.safetensors

GPU after SFT weight load: 10.1 GB
Trainable params (after SFT load — should be unchanged from above):
trainable params: 39,403,520 || all params: 7,980,504,352 || trainable%: 0.4937


In [6]:
# Verify SFT weights actually got copied (not just counted as "loaded")
# Pick one LoRA weight tensor and confirm it's not zero-init
for name, param in model.named_parameters():
    if "lora_A" in name and "vision_tower.encoder.layers.0" in name:
        print(f"{name}")
        print(f"  shape: {tuple(param.shape)}")
        print(f"  abs mean: {param.abs().mean().item():.6f}")
        print(f"  abs max:  {param.abs().max().item():.6f}")
        break

base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_A.default.weight
  shape: (16, 768)
  abs mean: 0.018288
  abs max:  0.037852


In [7]:
%uv pip freeze > /mnt/civicinsight/requirements-exp4c-WORKING.txt
!echo "Saved $(wc -l < /mnt/civicinsight/requirements-exp4c-WORKING.txt) packages to requirements-exp4c-WORKING.txt"


Using Python 3.12.6 environment at: /usr/local
Note: you may need to restart the kernel to use updated packages.
Saved 238 packages to requirements-exp4c-WORKING.txt


In [8]:
import random
import re

# perturbation library. a collection of helper functions to
# manipulate our gold text for DPO
MARKER = "[civicinsight-v1]"
BANNED_HEDGES = ["around ", "approximately ", "roughly ", "about "]
TOOLTIP_TEXT = "tooltip"
NO_TOOLTIP_TEXT = ". No tooltip is visible."
OPENER_SLOT = "This "

def strip_marker(gold: str) -> str:             # returns the gold string with the MARKER removed
    return gold.replace(f"{MARKER} ", "")

def inject_hedge(gold: str) -> str:
    # find number in string
    num = re.search(r"\b\d+\b", gold)
    if not num:
        return gold
    hedge = random.choice(BANNED_HEDGES)
    return gold[:num.start()] + hedge + gold[num.start():]

def break_consistency(gold: str) -> str:
    # look up the TOOLTIP_TEXT string
    if TOOLTIP_TEXT not in gold.lower():
        # if not found return gold
        return gold
    # if found, append NO_TOOLTIP_TEXT at end and return
    return gold.rstrip(".") + NO_TOOLTIP_TEXT

def wrong_slot(gold: str) -> str:
    """Drift the slot opener. Targets exp4-results.md's 'A [type]' / no-article failures."""
    if OPENER_SLOT not in gold:
        return gold

    # Randomly either:
    #   replace with "" so gold becomes "[civicinsight-v1] box plot"
    #   replace with "A" so "[civicinsight-v1] A box plot"
    return gold.replace(OPENER_SLOT, random.choice(["A ", ""]), 1)

def positional_schema_swap(gold: str) -> str:
    """Swap first two percentage values. Targets stacked-bar positional binding bug."""
    matches = list(re.finditer(r"\b(\d{1,3})%", gold))   # find all numbers with %
    if len(matches) < 2:                             # we need at least 2 percentage numbers to swap with, otherwise nothing to swap
        return gold

    m1, m2 = matches[0], matches[1]
    v1 = m1.group(1)                                     # first percent value, e.g., "53"
    v2 = m2.group(1)                                     # second percent value e.g., "80"
    return (
        gold[:m1.start(1)]                               # gold upto v1's start
        + v2                                             # swap v2 into v1
        + gold[m1.end(1):m2.start(1)]                    # string between m1 and m2
        + v1                                             # v1 in v2's spot
        + gold[m2.end(1):]                               # the rest after v2
    )

def googly_wrap(gold: str) -> str:
    """Wrap in **bold** headers + bullets. Easy negative for style regression."""
    body = gold.replace(MARKER, "").strip()
    return f"**Aria Label:**\n\n* {body}\n\nThis description is provided for screen readers."

PERTURBATIONS = {
    "strip_marker": strip_marker,
    "wrong_slot": wrong_slot,
    "inject_hedge": inject_hedge,
    "positional_schema_swap": positional_schema_swap,
    "break_consistency": break_consistency,
    "googly_wrap": googly_wrap,
}

# sanity check — see each perturbation applied to a sample gold
sample = "[civicinsight-v1] This stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box."
print(f"GOLD: {sample}\n")
for name, fn in PERTURBATIONS.items():
    print(f"[{name}]")
    print(f"  {fn(sample)}\n")


GOLD: [civicinsight-v1] This stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box.

[strip_marker]
  This stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box.

[wrong_slot]
  [civicinsight-v1] A stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box.

[inject_hedge]
  [civicinsight-v1] This stacked bar chart shows USA at about 80% urban and 19% rural areas. Tooltip is open for the Gauche box.

[positional_schema_swap]
  [civicinsight-v1] This stacked bar chart shows USA at 19% urban and 80% rural areas. Tooltip is open for the Gauche box.

[break_consistency]
  [civicinsight-v1] This stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box. No tooltip is visible.

[googly_wrap]
  **Aria Label:**

* This stacked bar chart shows USA at 80% urban and 19% rural areas. Tooltip is open for the Gauche box.

This description 

In [9]:
PROMPT = "Generate an aria-label for this data visualization image."
DATASET_JSON = "/mnt/civicinsight/training/dataset.marked.json"
STD_TRAIN_IMAGE_DIR = "/mnt/civicinsight/standardized"
ROTATION = ["strip_marker", "wrong_slot", "inject_hedge", "break_consistency", "positional_schema_swap", "googly_wrap"]

def make_pair(image, gold, rejected, prompt=PROMPT):
    return {
        "images": [image],
        "prompt": prompt,
        "chosen": gold,
        "rejected": rejected,
    }

pairs = []
with open(DATASET_JSON) as f:
    dataset = json.load(f)

from collections import Counter
counter = Counter()

for i, record in enumerate(dataset):
    img = Image.open(os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"])))
    gold = record["aria_label"]
    pert_name = ROTATION[i % len(ROTATION)]
    rejected = PERTURBATIONS[pert_name](gold)

    if rejected != gold:
        pairs.append(make_pair(img, gold, rejected))
        # inside the loop, after rejected != gold check:
        counter[pert_name] += 1

# Over-represent positional_schema_swap on golds with ≥2 percent values
# (rotation undersamples this — it's our highest-priority bug fix target)
for record in dataset:
    img = Image.open(os.path.join(STD_TRAIN_IMAGE_DIR, os.path.basename(record["image"])))
    gold = record["aria_label"]
    rejected = positional_schema_swap(gold)
    if rejected != gold:  # only if perturbation actually swapped (i.e., ≥2 %)
        pairs.append(make_pair(img, gold, rejected))
        counter["positional_schema_swap"] += 1

print(f"Total preference pairs: {len(pairs)}")
sample = pairs[0]
print(f"\n=== Sample pair (text only) ===")
print(f"Prompt:   {sample['prompt']}")
print(f"Chosen:   {sample['chosen'][:200]}...")
print(f"Rejected: {sample['rejected'][:200]}...")
print(f"Image:    {type(sample['images'][0]).__name__}")

# after loop:
print(f"\nPerturbation distribution:")
for name, count in counter.items():
    print(f"  {name}: {count}")


Total preference pairs: 52

=== Sample pair (text only) ===
Prompt:   Generate an aria-label for this data visualization image.
Chosen:   [civicinsight-v1] This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' shows global CO2 emissions. The X-axis shows years from 1880 to 2020. The Y-axis s...
Rejected: This line chart titled 'Émissions mondiales de CO2 par type de carburant et par secteur d'activité' shows global CO2 emissions. The X-axis shows years from 1880 to 2020. The Y-axis shows millions of m...
Image:    PngImageFile

Perturbation distribution:
  strip_marker: 9
  wrong_slot: 9
  inject_hedge: 8
  break_consistency: 6
  positional_schema_swap: 12
  googly_wrap: 8


In [10]:
# ============================================================
# Sweep the Held-out — run pre-DPO inference on EVERY image in the test dir
# ============================================================
# switch model to eval
model.eval()

STD_TEST_IMG_DIR = "/mnt/civicinsight/test"
prompt = "Generate an aria-label for this data visualization image."

test_images = sorted([f for f in os.listdir(STD_TEST_IMG_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))])
print(f"Found {len(test_images)} held-out images\n")

before_dpo_results = {}


for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"IMAGE: {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

# eval is done, back to training mode
model.train()

Found 5 held-out images

IMAGE: baseline-1.png | 128.2s
user
Generate an aria-label for this data visualization image.
model
[civicinsight-v1] This choropleth map shows real estate prices per square metre in French communes. The color scale ranges from 0 - 1076 (light pink) to 2989 - 6571 (dark red), with steps of 1 000. The map is centered on Plœrières, which is currently selected with a tooltip showing a price of 2643 EUR/m2. The legend on the right shows the color scale: dark blue (top) is 0 - 1 076, light blue is 1 076 - 1 394, green is 1 394 - 1 568, yellow is 1 568 - 1 730, pink is 1 730 - 1 944, light red is 1 944 - 2 187, pink-red is 2 187 - 2 560, red is 2 560 - 2 989, and dark red (bottom) is 2 989 - 6 571. The map also shows commune names: Ajaccio, Plouguer, Rostrenen, Ste-Anne-du-Faou, Gourin, Cléguérec, Pontivy, Loudéac, Saint-Méen-le-Grand, Mauron, Scorien, Le Faouët, Quimperlé, Hennebont, Lorient, Auray, Vannes, Saint-Gildas-de-Rheux, Quiberon, Herbignac, Plœrmel, Guer, 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma4ForConditionalGeneration(
      (model): Gemma4Model(
        (vision_tower): Gemma4VisionModel(
          (patch_embedder): Gemma4VisionPatchEmbedder(
            (input_proj): Linear(in_features=768, out_features=768, bias=False)
          )
          (encoder): Gemma4VisionEncoder(
            (rotary_emb): Gemma4VisionRotaryEmbedding()
            (layers): ModuleList(
              (0-15): 16 x Gemma4VisionEncoderLayer(
                (self_attn): Gemma4VisionAttention(
                  (q_proj): Gemma4ClippableLinear(
                    (linear): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=False)
                      (lora_dropout): ModuleDict(
                        (default): Identity()
                      )
                      (lora_A): ModuleDict(
                        (default): Linear(in_features=768, out_features=16, bias=False)
               

In [11]:
import os
os.makedirs("/mnt/civicinsight/hf_cache", exist_ok=True)
os.environ["HF_DATASETS_CACHE"] = "/mnt/civicinsight/hf_cache"
print(f"Cache set to: {os.environ['HF_DATASETS_CACHE']}")
print(f"  exists: {os.path.exists(os.environ['HF_DATASETS_CACHE'])}")
print(f"  writable: {os.access(os.environ['HF_DATASETS_CACHE'], os.W_OK)}")

Cache set to: /mnt/civicinsight/hf_cache
  exists: True
  writable: True


In [12]:
from datasets import Dataset, Features, Image as HFImage, Value
import time

# Build dataset with explicit Image feature type
features = Features({
    "images": [HFImage()],   # HF native image type, handles serialization
    "prompt": Value("string"),
    "chosen": Value("string"),
    "rejected": Value("string"),
})

start = time.time()
pairs_ds_typed = Dataset.from_list(pairs, features=features)
print(f"Built typed dataset in {time.time()-start:.1f}s")
print(f"Columns: {pairs_ds_typed.column_names}")
print(f"First image: {type(pairs_ds_typed[0]['images'][0])}")

Built typed dataset in 0.0s
Columns: ['images', 'prompt', 'chosen', 'rejected']
First image: <class 'PIL.PngImagePlugin.PngImageFile'>


In [ ]:
# DPO training run — main event
# Replaces SFTTrainer with DPOTrainer; uses preference pairs from cell 7

from datasets import Dataset
pairs_ds = Dataset.from_list(pairs)
print(f"Converted {len(pairs)} pairs to Dataset")

dpo_trainer = DPOTrainer(
    model=model,                                    # policy (will be trained)
    ref_model=None,                                 # ref_model=None → DPOTrainer auto-creates frozen reference from current model state
    processing_class=tokenizer,                     # if newer TRL errors with TypeError: swap to processing_class=tokenizer
    train_dataset=pairs_ds_typed,                         # preference pairs from cell 7, converted to Dataset
    args=DPOConfig(
        beta=0.1,                                   # KL penalty strength (DPO-specific; standard starting point)
        per_device_train_batch_size=1,              # keep simple for v1; A100 has headroom for more in v2
        gradient_accumulation_steps=4,              # effective batch of 4
        num_train_epochs=3,                         # DPO converges faster than SFT (1-3 epochs typical)
        learning_rate=5e-6,                         # ~40x lower than SFT's 2e-4 (standard for DPO)
        output_dir="/mnt/civicinsight/checkpoints/exp4c-dpo",
        max_length=2048,                            # max tokens per example (image + text combined)
        max_prompt_length=1024,                     # DPO-specific: max tokens for the prompt portion
        remove_unused_columns=False,                # required for image+text format
        logging_steps=1,                            # log every step (small dataset, want fine-grained loss curve)
        dataset_num_proc=1,                         # Modal reports os.cpu_count()=47 but allocates ~2 cores;
                                                    # default explodes RAM (40GB+) and stalls. Keep small.
    ),
)

print(f"GPU before DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print(f"Total preference pairs: {len(pairs)}")
print(f"\nStarting DPO training...")
dpo_trainer.train()
print(f"\nDPO training ran without crash!")
print(f"GPU after DPO: {torch.cuda.memory_allocated()/1e9:.1f} GB")


Converted 52 pairs to Dataset


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

In [ ]:
# AFTER-DPO held-out sweep — same 5 images, same prompt, same do_sample=False as cell 8
# Direct comparison anchor against before_dpo_results
model.eval()

after_dpo_results = {}
for img_name in test_images:
    img_path = os.path.join(STD_TEST_IMG_DIR, img_name)
    img = Image.open(img_path)
    message = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT},
    ]}]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=600, do_sample=False)
    elapsed = time.time() - start
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    after_dpo_results[img_name] = decoded
    print(f"{'='*70}")
    print(f"AFTER-DPO | {img_name} | {elapsed:.1f}s")
    print(f"{'='*70}")
    print(decoded)
    print()

In [ ]:
# Extended scorecard — existing 3 metrics + 2 NEW (positional-schema, consistency)
# Compares before-DPO vs after-DPO across all 5 held-outs

# Existing (copied from 07-experiment-4 cell 12)
SLOT_OPENERS = ("This line", "This bar", "This stacked", "This box", "This scatter",
                "This choropleth", "This hexagonal", "This panel", "This small multiples",
                "This untitled", "This area", "This pie", "This gauge", "This table")
BANNED_ADJECTIVES = ("around ", "approximately", "roughly ", "about ", "appears to",
                     "notably", "dramatically", "rises steadily", "drops steadily",
                     "significantly", "suggesting")

# NEW: positional-schema known-wrong segment-value bindings per held-out image
# format: list of (label_substr, wrong_value_substr) — if BOTH appear within 100 chars, flag
POSITIONAL_KNOWN_WRONGS = {
    "rural-vs-urban.png": [
        ("China", "53%"),      # China capital ~7-10%; the 53% is on other-urban
        ("USA", "80%"),        # USA capital ~10%; the 80% is actually on the other-urban segment
        ("Australia", "88%"),  # Australia capital ~8%; the 88% is on other-urban
        ("India", "30%"),      # India capital ~2%; the 30% is on the other-urban segment
    ],
}

# NEW: internal-consistency contradiction patterns (regex pairs that should NOT both appear in same output)
CONSISTENCY_CONTRADICTIONS = [
    (r"tooltip\s+is\s+visible", r"no\s+tooltip\s+is\s+visible"),
    (r"is\s+selected", r"no\s+\w+\s+is\s+selected"),
]

def score(output, img_name=None):
    assistant_part = output.split("model\n")[-1] if "model\n" in output else output
    assistant_part = assistant_part.strip()
    lower = assistant_part.lower()

    has_marker = MARKER in assistant_part
    opens_with_slot = any(
        assistant_part.lstrip().startswith(MARKER + " " + s)
        or assistant_part.lstrip().startswith(s)
        for s in SLOT_OPENERS
    )
    banned_found = [a.strip() for a in BANNED_ADJECTIVES if a in lower]

    # NEW: positional-schema (only flagged for held-outs in POSITIONAL_KNOWN_WRONGS)
    positional_errors = []
    if img_name and img_name in POSITIONAL_KNOWN_WRONGS:
        for s1, s2 in POSITIONAL_KNOWN_WRONGS[img_name]:
            if s1 in assistant_part and s2 in assistant_part:
                idx1 = assistant_part.index(s1)
                idx2 = assistant_part.index(s2)
                if abs(idx1 - idx2) < 100:  # within 100 chars = likely paired
                    positional_errors.append(f"{s1}+{s2}")

    # NEW: internal consistency
    consistency_violations = []
    for p1, p2 in CONSISTENCY_CONTRADICTIONS:
        if re.search(p1, lower) and re.search(p2, lower):
            consistency_violations.append(f"{p1} AND {p2}")

    return {
        "has_marker": has_marker,
        "opens_with_slot": opens_with_slot,
        "banned_adjectives_found": banned_found,
        "positional_errors": positional_errors,
        "consistency_violations": consistency_violations,
    }

def print_scorecard(results, label):
    print(f"\n{'='*100}")
    print(f"SCORECARD — {label}")
    print(f"{'='*100}")
    print(f"{'image':<45} {'marker':<8} {'slot':<6} {'banned':<25} {'pos-err':<15} {'consist'}")
    print("-" * 100)
    for img_name, out in results.items():
        s = score(out, img_name)
        banned = ",".join(s["banned_adjectives_found"]) or "none"
        pos = ",".join(s["positional_errors"]) or "none"
        consist = "FAIL" if s["consistency_violations"] else "ok"
        print(f"{img_name:<45} {str(s['has_marker']):<8} {str(s['opens_with_slot']):<6} {banned:<25} {pos:<15} {consist}")

    n = len(results)
    marker_rate = sum(1 for o in results.values() if score(o)['has_marker']) / n
    slot_rate = sum(1 for o in results.values() if score(o)['opens_with_slot']) / n
    no_adj_rate = sum(1 for o in results.values() if not score(o)['banned_adjectives_found']) / n
    no_pos_err = sum(1 for img_name, o in results.items() if not score(o, img_name)['positional_errors']) / n
    no_consist_viol = sum(1 for o in results.values() if not score(o)['consistency_violations']) / n
    print(f"\nAggregate ({n} held-outs):")
    print(f"  marker:                 {marker_rate:.0%}")
    print(f"  slot opener:            {slot_rate:.0%}")
    print(f"  no banned adjectives:   {no_adj_rate:.0%}")
    print(f"  no positional errors:   {no_pos_err:.0%}    [NEW]")
    print(f"  no consistency issues:  {no_consist_viol:.0%}    [NEW]")

print_scorecard(before_dpo_results, "BEFORE DPO (Exp 4b SFT only)")
print_scorecard(after_dpo_results, "AFTER DPO (Exp 4c SFT+DPO)")


In [ ]:
# Save full results JSON to Modal Volume (mirrors Exp 4b cell 16)
artifact = {
    "experiment": "exp4c-dpo",
    "date": "2026-04-25",
    "config": {
        "starting_checkpoint": SFT_CHECKPOINT,
        "beta": 0.1,
        "lr": 5e-6,
        "epochs": 3,
        "preference_pair_count": len(pairs),
    },
    "before_dpo": before_dpo_results,
    "after_dpo": after_dpo_results,
}

os.makedirs("/mnt/civicinsight/results", exist_ok=True)
with open("/mnt/civicinsight/results/exp4c-results.json", "w") as f:
    json.dump(artifact, f, indent=2, ensure_ascii=False)
print("Saved to /mnt/civicinsight/results/exp4c-results.json")


In [ ]:
# Push DPO adapter to HF private repo as a new commit

REPO_ID = "shahfazal/civicinsight-gemma4-e4b-it-exp4c"
DPO_CHECKPOINT_DIR = "/mnt/civicinsight/checkpoints/exp4c-dpo"

create_repo(REPO_ID, private=True, repo_type="model", exist_ok=True)

# find the most recent checkpoint dir (DPO might write multiple checkpoints over epochs)
checkpoint_dirs = sorted(
    [d for d in os.listdir(DPO_CHECKPOINT_DIR) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1]),
)
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoint dirs found in {DPO_CHECKPOINT_DIR}")

final_checkpoint = os.path.join(DPO_CHECKPOINT_DIR, checkpoint_dirs[-1])
print(f"Pushing checkpoint: {final_checkpoint}")

# fix base_model metadata in README before push (same workaround as Exp 4b)
readme_path = os.path.join(final_checkpoint, "README.md")
if os.path.exists(readme_path):
    with open(readme_path) as f:
        content = f.read()
    content = content.replace(
        "base_model: /mnt/civicinsight/model",
        "base_model: unsloth/gemma-4-E4B-it",
    )
    with open(readme_path, "w") as f:
        f.write(content)

api = HfApi()
api.upload_folder(
    folder_path=final_checkpoint,
    repo_id=REPO_ID,
    repo_type="model",
    ignore_patterns=["optimizer.pt", "scheduler.pt", "rng_state.pth"],
    commit_message="Exp 4c DPO — preference learning on Exp 4b base",
)
print(f"\nPushed to https://huggingface.co/{REPO_ID}")
